In [1]:
# 1. Install Library
!pip install -q transformers datasets accelerate torch torchvision scikit-learn

In [2]:
import os
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from datasets import load_dataset, DatasetDict
from transformers import (
    ViTForImageClassification,
    ViTImageProcessor,
    TrainingArguments,
    Trainer,
    default_data_collator
)
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, confusion_matrix
import shutil
from google.colab import files
import json

In [3]:
# --- SETTING SKENARIO 1 (BASELINE) ---
MODEL_CHECKPOINT = "google/vit-base-patch16-224"
OUTPUT_DIR = "./vit_scenario_1_baseline"
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 20
LEARNING_RATE = 3e-5
SEED = 42

# Matikan Fitur Advanced
LABEL_SMOOTHING = 0.0
DROPOUT = 0.0

In [4]:
# --- 2. Setup Kaggle & Download Data ---
# Cek apakah kaggle.json sudah ada, jika belum minta upload
if not os.path.exists('kaggle.json'):
    print("Upload file kaggle.json kamu:")
    files.upload()

# Pindahkan ke folder sistem
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download Dataset
print("\nSedang mendownload dataset...")
!kaggle datasets download -d fanconic/skin-cancer-malignant-vs-benign --force

# Unzip
if os.path.exists('skin-cancer-malignant-vs-benign.zip'):
    print("Mengekstrak data...")
    !unzip -q -o skin-cancer-malignant-vs-benign.zip -d dataset_binary
    DATA_DIR = "/content/dataset_binary"
    print("✅ Dataset Siap!")
else:
    print("❌ Gagal download dataset.")

Upload file kaggle.json kamu:


Saving kaggle.json to kaggle.json

Sedang mendownload dataset...
Dataset URL: https://www.kaggle.com/datasets/fanconic/skin-cancer-malignant-vs-benign
License(s): unknown
100% 325M/325M [00:02<00:00, 145MB/s]

Mengekstrak data...
✅ Dataset Siap!


In [5]:
# --- 3. Load & Split Dataset (DENGAN DATA VALIDASI) ---
# Muat data asli dari folder Kaggle
kaggle_train = load_dataset("imagefolder", data_dir=os.path.join(DATA_DIR, "train"), split="train")
test_ds      = load_dataset("imagefolder", data_dir=os.path.join(DATA_DIR, "test"), split="train")

# INI BAGIAN YANG DIUBAH: Pecah data latih Kaggle menjadi Latih murni (80%) dan Validasi (20%)
train_val_split = kaggle_train.train_test_split(test_size=0.2, seed=SEED)
train_ds = train_val_split['train']
val_ds   = train_val_split['test']

print(f"Jumlah Data Latih (Training)  : {len(train_ds)}")
print(f"Jumlah Data Validasi (Validation): {len(val_ds)}")
print(f"Jumlah Data Uji (Test)        : {len(test_ds)}")

Resolving data files:   0%|          | 0/2637 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Resolving data files:   0%|          | 0/660 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Jumlah Data Latih (Training)  : 2109
Jumlah Data Validasi (Validation): 528
Jumlah Data Uji (Test)        : 660


In [6]:
# Load Processor
processor = ViTImageProcessor.from_pretrained(MODEL_CHECKPOINT)

def preprocess_train(batch):
    inputs = processor([x for x in batch['image']], return_tensors='pt')
    inputs['label'] = batch['label']
    return inputs

def preprocess_eval(batch):
    inputs = processor([x for x in batch['image']], return_tensors='pt')
    inputs['label'] = batch['label']
    return inputs

# Terapkan transformasi ke dataset
print("Menerapkan Transformasi (Resize & Normalize)...")
train_ds.set_transform(preprocess_train)
val_ds.set_transform(preprocess_eval)
test_ds.set_transform(preprocess_eval)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

Menerapkan Transformasi (Resize & Normalize)...


In [7]:
# --- 4. Load Model (Skenario 1: ViT Standard - Pure Baseline) ---
model = ViTForImageClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=2,
    id2label={0: "benign", 1: "malignant"},
    label2id={"benign": 0, "malignant": 1},
    ignore_mismatched_sizes=True
)

# Matikan Dropout Config
model.config.hidden_dropout_prob = DROPOUT
model.config.attention_probs_dropout_prob = DROPOUT

# 1. BEKUKAN (FREEZE) SELURUH BACKBONE:
# Ini membuat bobot asli dari ImageNet tidak ikut dilatih/berubah.
for param in model.vit.parameters():
    param.requires_grad = False

# 2. AKTIFKAN HANYA LAPISAN TERAKHIR (CLASSIFIER HEAD):
# Ini adalah satu-satunya bagian yang akan dilatih untuk menebak Kanker/Bukan.
for param in model.classifier.parameters():
    param.requires_grad = True

print("✅ Model Baseline NO Fine Tuning Siap!")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                        
------------------+----------+----------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([2])          
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([2, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


✅ Model Baseline NO Fine Tuning Siap!


In [8]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    # Hitung Akurasi & F1
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='weighted')

    return {"accuracy": acc, "f1": f1}

In [9]:
# --- 6. Training Arguments (Standar Baseline) ---
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.0,
    eval_strategy="epoch",       # Evaluasi tiap epoch
    save_strategy="epoch",       # Simpan tiap epoch
    logging_steps=50,
    load_best_model_at_end=True, # Simpan model dengan metric terbaik
    metric_for_best_model="accuracy",
    save_total_limit=1,
    remove_unused_columns=False,
    report_to="none",
    fp16=torch.cuda.is_available()
)

In [10]:
# --- 7. Inisialisasi Trainer ---
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=processor,
    compute_metrics=compute_metrics,
    data_collator=default_data_collator
)

In [11]:
# --- 8. MULAI TRAINING ---
print("Memulai Training Skenario 1 (Pure Baseline - Standard ViT)...")
train_result = trainer.train()
print("Selesai. Metrics:", train_result.metrics)

# Evaluasi Akhir
print("\n--- MENGUJI MODEL TERBAIK DI DATA UJI (TEST SET) ---")
final_metrics = trainer.predict(test_ds)
print("Akurasi Final Skenario 1:", final_metrics.metrics['test_accuracy'])
print("F1-Score Final Skenario 1:", final_metrics.metrics['test_f1'])

Memulai Training Skenario 1 (Pure Baseline - Standard ViT)...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.709075,0.651664,0.651515,0.618985
2,0.646424,0.590568,0.706439,0.695894
3,0.592726,0.547095,0.753788,0.746276
4,0.531772,0.512471,0.780303,0.776462
5,0.498915,0.486612,0.801136,0.798603
6,0.510725,0.465352,0.804924,0.802245
7,0.463515,0.449096,0.818182,0.815954
8,0.444522,0.434863,0.831439,0.830440
9,0.442328,0.423959,0.835227,0.834368
10,0.428838,0.415561,0.839015,0.838285


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Selesai. Metrics: {'train_runtime': 570.2716, 'train_samples_per_second': 73.965, 'train_steps_per_second': 2.315, 'total_flos': 3.2686121219434906e+18, 'train_loss': 0.45728583588744653, 'epoch': 20.0}

--- MENGUJI MODEL TERBAIK DI DATA UJI (TEST SET) ---


Akurasi Final Skenario 1: 0.8303030303030303
F1-Score Final Skenario 1: 0.8302542934603635


In [12]:
# --- 9. Menyimpan & Mendownload Model ---
nama_zip = "vit_baseline_pure"
print(f"📦 Mengompres {OUTPUT_DIR}...")
shutil.make_archive(nama_zip, 'zip', OUTPUT_DIR)

print("⬇️ Mendownload Model...")
try:
    files.download(f"{nama_zip}.zip")
except:
    pass

📦 Mengompres ./vit_scenario_1_baseline...
⬇️ Mendownload Model...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
# --- 10. Menyimpan Metrik JSON ---
SCENARIO_NAME = "Skenario 1 (Baseline ViT)"
FILENAME = "metrics_scenario_1.json"

def save_training_metrics(trainer, test_ds, scenario_name, filename):
    print(f"\n📊 Sedang memproses histori untuk {scenario_name}...")

    # Ambil History Training (Loss & Accuracy per Epoch)
    log_history = trainer.state.log_history
    epochs_val, val_loss, val_acc = [], [], []
    train_loss = []

    for entry in log_history:
        if 'loss' in entry: # Training log
            train_loss.append(entry['loss'])
        elif 'eval_loss' in entry: # Validation log
            epochs_val.append(entry['epoch'])
            val_loss.append(entry['eval_loss'])
            val_acc.append(entry['eval_accuracy'])

    # Hitung Metrik Final di Data Test
    print("🧠 Mengekstrak performa final & Confusion Matrix di test set...")
    predictions = trainer.predict(test_ds)
    preds = np.argmax(predictions.predictions, axis=1)
    labels = predictions.label_ids

    final_acc = accuracy_score(labels, preds)
    final_f1 = f1_score(labels, preds, average='weighted')

    # Hitung Confusion Matrix
    cm = confusion_matrix(labels, preds)
    cm_list = cm.tolist()

    # Susun Dictionary
    data = {
        "name": scenario_name,
        "epochs": epochs_val,
        "train_loss": train_loss[:len(val_loss)],
        "val_loss": val_loss,
        "val_acc": val_acc,
        "final_test_accuracy": final_acc,
        "final_test_f1": final_f1,
        "confusion_matrix": cm_list
    }

    # Simpan ke JSON
    with open(filename, "w") as f:
        json.dump(data, f)

    print(f"✅ Data {scenario_name} berhasil disimpan ke '{filename}'")
    try:
        files.download(filename)
    except:
        pass

save_training_metrics(trainer, test_ds, SCENARIO_NAME, FILENAME)


📊 Sedang memproses histori untuk Skenario 1 (Baseline ViT)...
🧠 Mengekstrak performa final & Confusion Matrix di test set...


✅ Data Skenario 1 (Baseline ViT) berhasil disimpan ke 'metrics_scenario_1.json'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>